# Self-Attention
### When every word gets to look at every other word — including itself

---

## 📖 The Story

It is 2017. Attention has already improved translation (Topic 1), but it is still bolted onto a slow, sequential RNN. A team at Google Brain asks: *"What if attention wasn't just a helper — what if it WAS the entire model?"*

To make this work, every word in a sentence needs to look at every OTHER word in that same sentence — not just one sequence looking at another. This notebook builds that exact mechanism: **self-attention**, the core building block behind "Attention Is All You Need," GPT, and BERT.

---

**What this notebook covers:**
- Building Query, Key, Value projections from scratch
- Implementing scaled dot-product self-attention with NumPy
- Visualizing how tokens attend to each other within one sentence
- Verifying our implementation against PyTorch
- Demonstrating WHY the √dₖ scaling factor is necessary

**Prerequisites:**
- Why Attention Matters (Topic 1)
- Matrix multiplication and linear algebra basics
- NumPy fundamentals

**Note:** Like Topic 1, this notebook uses an in-notebook example sentence rather than an external dataset — self-attention is best understood through direct, inspectable computation on a small example.

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — self-attention from scratch
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Heatmap visualizations
import torch                              # PyTorch — verification
import torch.nn.functional as F          # Scaled dot-product attention
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                        # Reproducibility
torch.manual_seed(42)

print("All imports successful ✅")

## Part 1: Theory Recap

Five things to know before we write any code:

- **Self-attention = a sequence attends to itself**: Unlike Topic 1's cross-attention (decoder looking at encoder), every token in self-attention looks at every OTHER token in the SAME sequence, including itself.
- **Three roles per token: Query, Key, Value**: Each token's embedding is projected into a Query (what it's looking for), a Key (what it offers), and a Value (the information it carries) using separate learned weight matrices WQ, WK, WV.
- **The formula: Attention(Q,K,V) = softmax(QKᵀ/√dₖ)V**: QKᵀ computes pairwise similarity between all tokens, √dₖ scaling prevents extreme softmax values, and the result is a weighted blend of all Value vectors.
- **Scaling by √dₖ keeps softmax well-behaved**: Without it, large dot products in high dimensions push softmax into an extremely peaked state, causing vanishing gradients during training.
- **Output shape matches input shape**: Self-attention transforms a sequence of embeddings into a new sequence of the SAME length, where each position is now context-aware.

## Setting Up a Real Example Sentence

We use the sentence **"The cat sat on mat"** — the same sentence from Topic 1 — so you can directly compare cross-attention (Topic 1) with self-attention (this notebook).

Each word gets a simple embedding vector. In a real transformer these come from a learned embedding layer; here we use small random vectors so the math stays inspectable.

In [ ]:
# Define our example sentence
words = ["The", "cat", "sat", "on", "mat"]
seq_len   = len(words)
d_model   = 8   # embedding dimension
d_k       = 8   # Query/Key dimension (often d_model // num_heads, but single-head here)

# Create input embeddings — one vector per word
# In a real model these come from a trained embedding layer
X = np.random.randn(seq_len, d_model) * 0.5

print("Input embeddings X:")
print(f"Shape: {X.shape}  (seq_len={seq_len}, d_model={d_model})\n")
for word, vec in zip(words, X):
    print(f"  {word:6s}: {np.round(vec, 2)}")

## Part 2: Self-Attention From Scratch

We now build self-attention step by step:
1. Create learned weight matrices WQ, WK, WV
2. Project X into Q, K, V
3. Compute QKᵀ (raw attention scores)
4. Scale by √dₖ
5. Apply softmax row-wise
6. Multiply by V to get the final output

In [ ]:
# Step 1: Create learned weight matrices for Q, K, V projections
# In a real model these are learned via backpropagation.
# Here we initialize them randomly to simulate a trained state.
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_k) * 0.5

print("Weight matrix shapes:")
print(f"  W_Q: {W_Q.shape}")
print(f"  W_K: {W_K.shape}")
print(f"  W_V: {W_V.shape}")

# Step 2: Project input embeddings into Query, Key, Value
# INTERVIEW NOTE: Same input X produces three DIFFERENT outputs Q, K, V
# because each uses a different learned weight matrix.
Q = X @ W_Q   # shape: (seq_len, d_k)
K = X @ W_K   # shape: (seq_len, d_k)
V = X @ W_V   # shape: (seq_len, d_k)

print(f"\nQ shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print("\nEach word now has its own Query, Key, and Value vector.")

In [ ]:
def softmax(x, axis=-1):
    """
    Numerically stable softmax along a given axis.
    Subtract row max before exp to prevent overflow.
    """
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def self_attention(Q, K, V, d_k):
    """
    Scaled Dot-Product Self-Attention.
    Attention(Q, K, V) = softmax(QKᵀ / √dₖ) V

    Args:
        Q, K, V : Query, Key, Value matrices — shape (seq_len, d_k)
        d_k     : dimension of Key/Query vectors (for scaling)

    Returns:
        output  : new contextualized representations — shape (seq_len, d_k)
        weights : attention weight matrix — shape (seq_len, seq_len)
    """
    # Step 1: Compute raw scores — every query against every key
    # QKᵀ shape: (seq_len, seq_len) — entry (i,j) = how much token i attends to token j
    scores = Q @ K.T
    print(f"  Step 1 — Raw scores QKᵀ shape: {scores.shape}")

    # Step 2: Scale by √dₖ — prevents large values from saturating softmax
    # INTERVIEW NOTE: This is THE key difference from basic dot-product attention (Topic 1)
    scaled_scores = scores / np.sqrt(d_k)
    print(f"  Step 2 — Scaled by √{d_k} = {np.sqrt(d_k):.3f}")

    # Step 3: Softmax row-wise — each row sums to 1
    # Row i = how token i distributes its attention across all tokens
    weights = softmax(scaled_scores, axis=-1)
    print(f"  Step 3 — Softmax applied. Each row sums to: {weights.sum(axis=-1).round(4)}")

    # Step 4: Weighted sum of Values
    output = weights @ V
    print(f"  Step 4 — Output shape: {output.shape} (same as input — context-aware now)")

    return output, weights

print("Running self-attention on 'The cat sat on mat':\n")
output, attn_weights = self_attention(Q, K, V, d_k)

## What Just Happened?

We just computed self-attention for an entire sentence in 4 matrix operations.

- Every word produced its own Query, Key, and Value vector
- We compared every word's Query against every word's Key (QKᵀ) — this produced a 5×5 matrix since we have 5 words
- We scaled the scores by √dₖ to keep softmax well-behaved
- Softmax converted each row into a probability distribution — "sat" now has a distribution over how much it attends to "The", "cat", "sat" (itself), "on", "mat"
- The final output blends every word's Value vector according to these weights — "sat"'s new representation is now informed by the whole sentence

This single operation — repeated and stacked across many layers — is the entire engine behind BERT and GPT.

In [ ]:
# --- Visualisation 1: Self-Attention Heatmap ---
# Unlike Topic 1's heatmap (different input/output sequences),
# this heatmap is SQUARE — same words on both axes, because
# self-attention is the sequence attending to itself.

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    attn_weights,
    xticklabels=words,
    yticklabels=words,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Attention Weight'}
)
ax.set_title('Self-Attention Weight Matrix\n"The cat sat on mat" attending to itself',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Key (being attended TO)', fontsize=11)
ax.set_ylabel('Query (doing the attending)', fontsize=11)
plt.tight_layout()
plt.show()

print("Notice the matrix is SQUARE (5x5) — every word attends to every word,")
print("including itself. Compare this to Topic 1's heatmap which was rectangular")
print("(English words x French words) because that was cross-attention between")
print("two DIFFERENT sequences.")

In [ ]:
# --- Visualisation 2: Effect of √dₖ Scaling ---
# Directly demonstrates WHY scaling matters, using our actual Q, K matrices

raw_scores = Q @ K.T  # unscaled

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('No Scaling (÷1)', raw_scores / 1),
    (f'Correct Scaling (÷√{d_k}={np.sqrt(d_k):.2f})', raw_scores / np.sqrt(d_k)),
    ('Over-Scaling (÷50)', raw_scores / 50)
]

for ax, (title, scaled) in zip(axes, configs):
    weights = softmax(scaled, axis=-1)
    sns.heatmap(weights, xticklabels=words, yticklabels=words,
                annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, cbar=False, vmin=0, vmax=1)
    # Measure how "peaked" the distribution is using average max weight per row
    avg_max = weights.max(axis=-1).mean()
    ax.set_title(f'{title}\nAvg max weight per row: {avg_max:.3f}', fontsize=10, fontweight='bold')

plt.suptitle('Why √dₖ Scaling Matters — Comparing Softmax Sharpness',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("No scaling → softmax becomes very peaked (close to 1.0) → model nearly ignores other tokens")
print("Correct √dₖ scaling → balanced attention → gradients flow well during training")
print("Over-scaling → attention becomes too flat → model can't distinguish relevant tokens")

## Part 3: PyTorch Verification

We verify our scratch implementation using PyTorch's official `F.scaled_dot_product_attention` function — the exact operation used inside real transformer implementations (including PyTorch's own `nn.MultiheadAttention` internals).

In [ ]:
# Convert our NumPy Q, K, V to PyTorch tensors
# Shape required by F.scaled_dot_product_attention: (batch, seq_len, dim)
Q_torch = torch.tensor(Q, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, d_k)
K_torch = torch.tensor(K, dtype=torch.float32).unsqueeze(0)
V_torch = torch.tensor(V, dtype=torch.float32).unsqueeze(0)

# PyTorch's official scaled dot-product attention
# By default it scales by 1/√(E) where E is the last dimension of Q — matches our manual scaling
with torch.no_grad():
    output_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch)

output_torch_np = output_torch.squeeze(0).numpy()  # (seq_len, d_k)

print("=== Scratch vs PyTorch Verification ===\n")
print("Output (first row — representation of 'The'):")
print(f"  Scratch : {np.round(output[0], 5)}")
print(f"  PyTorch : {np.round(output_torch_np[0], 5)}")

max_diff = np.max(np.abs(output - output_torch_np))
print(f"\nMax absolute difference across full output: {max_diff:.2e}")
print("✅ Match — our scratch implementation matches PyTorch's official function" if max_diff < 1e-4
      else "❌ Mismatch — check scaling or matrix shapes")

## Part 4: Hyperparameter Experiments

Two experiments to build deeper intuition:

1. **Effect of dₖ (Key/Query dimension) on attention sharpness** — does increasing dimensionality change how peaked the attention becomes, even WITH scaling?
2. **Effect of sequence length on attention matrix size** — directly demonstrating the O(n²) complexity

In [ ]:
# Experiment 1: Vary d_k and observe attention weight distribution
d_k_values = [2, 4, 8, 16, 32, 64]
avg_max_weights = []

for dk_test in d_k_values:
    # Generate fresh random Q, K for this dimension
    Q_test = np.random.randn(seq_len, dk_test)
    K_test = np.random.randn(seq_len, dk_test)

    scores_test = (Q_test @ K_test.T) / np.sqrt(dk_test)  # with proper scaling
    weights_test = softmax(scores_test, axis=-1)
    avg_max_weights.append(weights_test.max(axis=-1).mean())

# Experiment 2: Attention matrix size vs sequence length (O(n²) demonstration)
seq_lengths_test = [5, 10, 20, 50, 100, 200]
matrix_sizes = [n * n for n in seq_lengths_test]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: d_k vs attention sharpness (with proper scaling applied)
axes[0].plot(d_k_values, avg_max_weights, marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].set_title('Effect of dₖ on Attention Sharpness\n(with √dₖ scaling applied — stays stable)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('dₖ (Query/Key dimension)', fontsize=11)
axes[0].set_ylabel('Average Max Attention Weight', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: O(n²) complexity
axes[1].plot(seq_lengths_test, matrix_sizes, marker='s', color='red', linewidth=2, markersize=8)
axes[1].set_title('Self-Attention Matrix Size — O(n²) Growth', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sequence Length (n)', fontsize=11)
axes[1].set_ylabel('Attention Matrix Entries (n²)', fontsize=11)
axes[1].grid(True, alpha=0.3)
for x, y in zip(seq_lengths_test, matrix_sizes):
    axes[1].annotate(f'{y:,}', (x, y), textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8)

plt.suptitle('Self-Attention Experiments: Dimension Stability & Quadratic Cost',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"With proper √dₖ scaling, attention sharpness stays roughly stable across dimensions.")
print(f"But the attention matrix size grows quadratically: a 200-token sequence needs")
print(f"{200*200:,} score computations — this is the famous O(n²) bottleneck of self-attention.")

## Common Mistakes Students Make

**Mistake 1: Thinking Q, K, V are three different inputs**
In self-attention, Q, K, and V all come from the SAME input X — they are three different LINEAR PROJECTIONS of the same embeddings, not three separate pieces of data. Only the weight matrices (WQ, WK, WV) differ.

**Mistake 2: Forgetting that softmax is applied row-wise, not globally**
Each row of the attention matrix must independently sum to 1 — representing one token's distribution of attention over all tokens. A common bug is accidentally applying softmax over the wrong axis, which breaks the probability interpretation entirely.

**Mistake 3: Believing self-attention understands word order**
Self-attention as shown here is **permutation invariant** — if you shuffled the words, you would get the same attention pattern (just reordered), because nothing here encodes position information. This is exactly why Positional Encoding (Topic 4) is required.

## Part 5: Interview Corner

**Question: Walk me through how self-attention allows tokens to interact with each other.**

This directly addresses the module's second learning outcome. Complete answer:

**Setup:** Every token starts as an independent embedding vector with no knowledge of context.

**Step 1 — Roles:** Each token's embedding is projected into three vectors: Query (what am I looking for), Key (what do I represent to others), Value (what information do I carry).

**Step 2 — Interaction:** Every token's Query is compared against EVERY token's Key via dot product. This is the literal mechanism of "interaction" — token A's query algebraically interacts with token B's key.

**Step 3 — Weighting:** Softmax converts these interaction scores into a probability distribution — how much should token A listen to each other token, including itself?

**Step 4 — Information exchange:** Token A's new representation becomes a weighted blend of every token's Value vector. Information has literally flowed from every token into token A's updated representation.

**The key insight:** This happens for ALL tokens SIMULTANEOUSLY, in parallel matrix operations — not sequentially like an RNN. After one self-attention layer, every token "knows about" every other token. Stack several layers, and the model builds increasingly rich, context-aware representations — this is exactly how BERT understands that "it" refers to "animal" in our earlier example.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **Self-attention = a sequence attending to itself**: Unlike Topic 1's cross-attention (two different sequences), Query, Key, AND Value all come from the SAME input sequence here.
- **Q, K, V are three learned projections of the same embedding**: WQ, WK, WV are separate weight matrices that let one input embedding play three specialized roles — asking, offering, and providing content.
- **The formula is Attention(Q,K,V) = softmax(QKᵀ/√dₖ)V**: QKᵀ measures pairwise token similarity, √dₖ scaling prevents softmax saturation, and the final weighted sum of V gives every token a context-aware representation.
- **Self-attention is O(n²) in sequence length**: Every token compares against every other token, so a sequence twice as long needs four times the computation — this is the central scalability challenge of transformers.
- **Self-attention has no built-in sense of order**: It is permutation invariant — shuffling input tokens just reorders the output identically. Positional encoding (next topic) is what injects order information into the model.